In [1]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("janiobachmann/bank-marketing-dataset")

print("Dataset path:", path)
print("Files:", os.listdir(path))

df = pd.read_csv(f"{path}/bank.csv", sep=",")

df.head()

Using Colab cache for faster access to the 'bank-marketing-dataset' dataset.
Dataset path: /kaggle/input/bank-marketing-dataset
Files: ['bank.csv']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [2]:
print("Dataset Shape:", df.shape)

print("\nFeature Data Types:")
print(df.dtypes)

print("\nDataset Information:")
df.info()

Dataset Shape: (11162, 17)

Feature Data Types:
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
deposit      object
dtype: object

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 

In [3]:
print("Target Distribution:")
print(df["deposit"].value_counts())

print("\nTarget Distribution (%):")
print(df["deposit"].value_counts(normalize=True) * 100)

Target Distribution:
deposit
no     5873
yes    5289
Name: count, dtype: int64

Target Distribution (%):
deposit
no     52.616019
yes    47.383981
Name: proportion, dtype: float64


In [4]:
print("Missing Values:")
print(df.isnull().sum())

print("\nNumber of Duplicate Rows:")
print(df.duplicated().sum())

Missing Values:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64

Number of Duplicate Rows:
0


In [ ]:
#df = df.drop_duplicates()

#print("Dataset shape after removing duplicates:", df.shape)

In [5]:
# Features
X = df.drop("deposit", axis=1)

# Target
y = df["deposit"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11162, 16)
y shape: (11162,)


In [6]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Categorical Features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30, # basicallly validation + testing is 30%
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Temporary (validation + testing) set:", X_temp.shape)

Training set: (7813, 16)
Temporary (validation + testing) set: (3349, 16)


In [8]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (7813, 16)
Validation set: (1674, 16)
Test set: (1675, 16)


In [9]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
deposit
no     52.617432
yes    47.382568
Name: proportion, dtype: float64

Validation target distribution:
deposit
no     52.628435
yes    47.371565
Name: proportion, dtype: float64

Test target distribution:
deposit
no     52.597015
yes    47.402985
Name: proportion, dtype: float64


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), #because median is more robust to outliers
    ("scaler", StandardScaler())
])

# categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [11]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (7813, 51)
Processed validation shape: (1674, 51)
Processed test shape: (1675, 51)


### FEATURE ENGINEERING DECISIONS:

1. Target:
   - 'y' is the target variable.
   - It represents a classification problem: yes/no.

2. Duplicates:
   - Duplicate rows were removed to avoid repeated observations
     having extra influence on the model.

3. Missing Values:
   - Numerical features: median imputation.
   - Categorical features: most frequent value imputation.

4. Categorical Features:
   - One-Hot Encoding was used.
   - handle_unknown='ignore' prevents errors if new categories
     appear in validation or test data.

5. Numerical Features:
   - StandardScaler was applied.
   - Scaling is useful for algorithms dependent on feature magnitude
     or distance, such as Logistic Regression, kNN, and SVM.

6. Data Splitting:
   - 70% training, 15% validation, and 15% test.
   - random_state=42 ensures reproducible results.
   - Stratification was used because the target classes are imbalanced.

7. Data Leakage Prevention:
   - The preprocessing pipeline was fitted only on the training data.
   - Validation and test sets were transformed using the training-fitted pipeline.

## **SVM without feature scaling**

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# numerical preprocessing but WITHOUT scaling
numerical_transformer_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer_unscaled, numerical_features),
        ("cat", categorical_transformer_unscaled, categorical_features)
    ]
)

X_train_unscaled = preprocessor_unscaled.fit_transform(X_train)
X_val_unscaled = preprocessor_unscaled.transform(X_val)

print("Unscaled training shape:", X_train_unscaled.shape)
print("Unscaled validation shape:", X_val_unscaled.shape)

Unscaled training shape: (7813, 51)
Unscaled validation shape: (1674, 51)


In [32]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm_unscaled = SVC(random_state=42)

svm_unscaled.fit(X_train_unscaled, y_train)

y_train_pred_svm_unscaled = svm_unscaled.predict(X_train_unscaled)
y_val_pred_svm_unscaled = svm_unscaled.predict(X_val_unscaled)

train_accuracy_svm_unscaled = accuracy_score(
    y_train,
    y_train_pred_svm_unscaled
)

val_accuracy_svm_unscaled = accuracy_score(
    y_val,
    y_val_pred_svm_unscaled
)

print("SVM without any scaling")
print(f"Training Accuracy: {train_accuracy_svm_unscaled:.4f}")
print(f"Validation Accuracy: {val_accuracy_svm_unscaled:.4f}")

SVM without any scaling
Training Accuracy: 0.7424
Validation Accuracy: 0.7378


## **SVM with feature scaling**

In [34]:
svm_scaled = SVC(random_state=42)

svm_scaled.fit(X_train_processed, y_train)

y_train_pred_svm_scaled = svm_scaled.predict(X_train_processed)
y_val_pred_svm_scaled = svm_scaled.predict(X_val_processed)

train_accuracy_svm_scaled = accuracy_score(
    y_train,
    y_train_pred_svm_scaled
)

val_accuracy_svm_scaled = accuracy_score(
    y_val,
    y_val_pred_svm_scaled
)

print("SVM with sscaling")
print(f"Training Accuracy:   {train_accuracy_svm_scaled:.4f}")
print(f"Validation Accuracy: {val_accuracy_svm_scaled:.4f}")

SVM with sscaling
Training Accuracy:   0.8769
Validation Accuracy: 0.8662


## Final Comparison

In [35]:
import pandas as pd

svm_comparison = pd.DataFrame({
    "Model": [
        "SVM Without Scaling",
        "SVM With Scaling"
    ],
    "Training Accuracy": [
        train_accuracy_svm_unscaled,
        train_accuracy_svm_scaled
    ],
    "Validation Accuracy": [
        val_accuracy_svm_unscaled,
        val_accuracy_svm_scaled
    ]
})

svm_comparison

,Model,Training Accuracy,Validation Accuracy
0,SVM Without Scaling,0.742352,0.737754
1,SVM With Scaling,0.876872,0.866189


SVM with feature scaling performed significantly better than SVM without scaling. We can see that training and validation accuracy increased. This shows that feature scaling is very important for SVM because SVM is sensitive to feature magnitudes and distances. So without scaling features with larger numerical ranges can have a greater influence on the model.